# 02 — Steering Vectors

**Objetivo (días 5-6):** calcular e inyectar un steering vector en GPT-2 y comparar outputs con y sin steering.

**Entregable:** notebook funcional con steering vector de escepticismo inyectado. Output antes/después visible con mínimo 5 prompts.

**Prerequisito:** haber completado el notebook 01 — se asume que sabes qué es el residual stream, `run_with_cache()` y los hooks.

## Diccionario de conceptos

---

**Representación lineal de features**

Hipótesis central del activation steering: los modelos de lenguaje codifican conceptos como *direcciones* en el espacio de activaciones. Si el modelo tiene un concepto de escepticismo, existe una dirección en R^768 tal que moverse en esa dirección hace que el modelo procese el texto de forma más escéptica. Esta hipótesis está respaldada empíricamente (Turner et al. 2023, Elhage et al. 2022) y justifica que una simple suma vectorial pueda modificar el comportamiento del modelo.

---

**Espacio de activaciones**

El espacio vectorial de dimensión `d_model` (768 en GPT-2) donde viven las representaciones internas del modelo. Cada token en cada capa tiene una representación — un punto en este espacio. Los conceptos son regiones o direcciones dentro de él. La interpretabilidad mecanicista estudia la geometría de este espacio.

---

**Superposición (superposition)**

Fenómeno por el que los modelos codifican más features que dimensiones. Un espacio de 768 dimensiones puede representar miles de conceptos usando direcciones casi-ortogonales. Esto implica que los conceptos no ocupan dimensiones individuales sino combinaciones de ellas. La representación lineal funciona gracias a la superposición: cada dirección puede ser suficientemente ortogonal a las demás para ser distinguible. (Elhage et al. 2022)

---

**Contrastive Activation Engineering (ActAdd)**

El método de Turner et al. 2023 para calcular steering vectors. La intuición: si quiero encontrar la dirección del concepto escepticismo, genero activaciones con prompts que expresan certeza y prompts que expresan duda. La diferencia entre los centroides de ambos grupos es una estimación de la dirección del concepto en el espacio de activaciones.

---

**Prompts contrastivos**

Pares de prompts que representan conceptos opuestos. Un conjunto expresa el concepto positivo (lo que queremos añadir) y otro el negativo (de lo que queremos alejarnos). Usar múltiples prompts por concepto y promediar reduce el ruido específico de cada frase y captura mejor la dirección del concepto.

---

**Centroide**

La media aritmética de un conjunto de vectores. En este contexto: el punto promedio en el espacio de activaciones de todos los prompts de un mismo concepto. El steering vector es la diferencia entre el centroide del concepto negativo y el centroide del concepto positivo.

---

**Steering vector**

Un vector en R^768 que apunta de un concepto a otro en el espacio de activaciones. Se calcula como `centroide_negativo - centroide_positivo`. Sumarlo al residual stream durante la inferencia empuja la representación activa hacia la zona del espacio asociada al concepto negativo.

---

**Alpha — coeficiente de intensidad**

Escalar que controla la magnitud del steering. El vector se suma como `alpha * steering_vector`. Un alpha muy pequeño produce un efecto imperceptible; un alpha muy grande rompe la coherencia del texto porque desplaza la representación demasiado lejos de las zonas que el modelo sabe interpretar. El rango óptimo depende de la capa y la norma del vector — típicamente entre 10 y 20 para GPT-2.

---

**Norma del steering vector**

La magnitud del vector (`torch.norm()`). Si es muy pequeña, los dos conceptos son casi indistinguibles para el modelo en esa capa — señal de que hay que probar otra capa. Si es muy grande, puede indicar que los prompts son semánticamente muy distintos por razones no relacionadas con el concepto buscado.

---

**Inyección en el residual stream**

El mecanismo de steering: sumar el vector al tensor del residual stream en una capa específica durante el forward pass. Las capas posteriores reciben ese tensor modificado y continúan el cómputo normalmente. El modelo no sabe que hubo intervención — procesa la información modificada como si fuera la representación natural del texto.

---

**Capa de inyección**

La capa del transformer donde se inyecta el steering vector. Las capas intermedias (8-12 de 12 en GPT-2) suelen dar los mejores resultados: las capas iniciales codifican sintaxis básica y aún no tienen semántica rica; las capas finales están enfocadas en la predicción del próximo token y el steering tiene menos efecto. El rango 8-12 es el punto donde el modelo tiene representaciones semánticas maduras.

---

**Generación autoregresiva**

El proceso de generar texto token a token: el modelo predice el siguiente token, lo añade a la secuencia, y repite. En cada paso se hace un forward pass completo. El hook de steering se activa en cada uno de esos forward passes, así que el efecto se aplica continuamente durante toda la generación.

---

**Hedging lingüístico**

Expresiones que indican incertidumbre o cautela: might, perhaps, it is unclear, some argue, allegedly, it's possible that. Es el efecto observable esperado de un steering vector de escepticismo que funciona correctamente.

---

**Reversibilidad**

Una ventaja clave del steering frente al fine-tuning: el efecto existe solo mientras el hook está registrado. Quitar el hook restaura el comportamiento original al instante, sin recargar el modelo. Los pesos nunca se modifican.

## 1. Setup

Este notebook construye sobre el 01. Mientras que en el 01 aprendimos a *leer* activaciones con `run_with_cache`, aquí aprendemos a *escribir* en ellas durante la inferencia.

El flujo completo:
1. Definir prompts contrastivos (certeza vs. escepticismo)
2. Extraer activaciones medias de cada conjunto en una capa intermedia
3. Calcular la dirección en el espacio de activaciones que separa ambos conceptos
4. Inyectar esa dirección durante la generación y observar el cambio de comportamiento

In [1]:
!pip install transformer_lens -q

In [3]:
import torch
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained('gpt2')
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (h

## 2. Prompts contrastivos

**Contrastive activation engineering** (Turner et al. 2023): para encontrar la dirección de calma/esperanza en el espacio de activaciones:

1. Genera activaciones con prompts que expresan **calma y esperanza** → calcula su centroide (`pos_mean`)
2. Genera activaciones con prompts que expresan **miedo y devastación** → otro centroide (`neg_mean`)
3. `steering_vector = pos_mean - neg_mean` apunta de miedo hacia calma

La convención: `positive_prompts` es el concepto que quieres **añadir**, `negative_prompts` es el concepto del que quieres **alejarte**. El vector suma el positivo y resta el negativo.

In [4]:
positive_prompts = [  # concepto que queremos añadir
    'The patient felt calm and at peace with the news',
    'I am hopeful and trust that everything will work out',
    'Despite the diagnosis, I feel relieved and supported,',
]

negative_prompts = [  # concepto del que queremos alejarnos
    'The patient felt terrified and could not breathe',
    'I am devastated and overwhelmed with fear',
    'This is the worst news I have ever received,',
]

## 3. Extraer activaciones medias por concepto

Dos decisiones de diseño que merece la pena entender:

**Que token tomamos:** el último token (`[0, -1, :]`). En modelos autorregresivos como GPT-2 el último token es el que va a predecir el siguiente, por lo tanto su representación en el residual stream integra toda la información contextual del prompt. Es el punto donde confluye el significado.

**Que capa usamos:** las capas intermedias (8-12 de 12 en GPT-2) suelen tener las representaciones semánticas más ricas. Las capas iniciales codifican sintaxis y tokens individuales. Las capas finales están enfocadas en la tarea de predicción del próximo token. El rango 8-12 es el punto dulce donde el modelo ha procesado suficiente contexto pero aún no ha colapsado hacia la distribución de salida.

In [5]:
LAYER = 10

def get_mean_activation(prompts, layer):
    activations = []
    for prompt in prompts:
        tokens = model.to_tokens(prompt)
        _, cache = model.run_with_cache(tokens)
        act = cache["resid_post", layer][0, -1, :]  # último token, shape (768,)
        activations.append(act)
    return torch.stack(activations).mean(dim=0)

pos_mean = get_mean_activation(positive_prompts, LAYER)
neg_mean = get_mean_activation(negative_prompts, LAYER)

print("Shape activación media:", pos_mean.shape)  # (768,)

Shape activación media: torch.Size([768])


## 4. Calcular el steering vector

**Geometría del steering vector:**

```
neg_mean (miedo) <--------- pos_mean (calma/esperanza)
                        ^
                        |
               steering_vector = pos_mean - neg_mean
```

El vector apunta de miedo hacia calma en el espacio de 768 dimensiones. Al sumarlo
al residual stream durante la inferencia, trasladamos la representación activa
hacia la zona del espacio donde el modelo vive cuando procesa tranquilidad y esperanza.

La **norma** del vector es un indicador de calidad: si es muy pequeña, los dos
conceptos son casi indistinguibles para el modelo en esa capa.

In [6]:
steering_vector = pos_mean - neg_mean
# No normalizamos: conservamos la magnitud original del vector para que el alpha
# sea comparable a la escala real de las activaciones de GPT-2.

print('Steering vector shape:', steering_vector.shape)
print('Norma:', steering_vector.norm().item())

Steering vector shape: torch.Size([768])
Norma: 84.38777923583984


## 5. Inyectar el vector con un hook

**Por qué no funciona `model.generate(hooks=...)`:**

TransformerLens ignora silenciosamente los kwargs desconocidos en `generate()`. El hook nunca se ejecuta.

**La solución:** bucle de generación manual. En cada paso llamamos a `model(tokens)` directamente, envuelto en `model.hooks()` como context manager. Así el hook se aplica en cada forward pass de la generación autoregresiva.

**Alpha — como elegirlo:**

| alpha | Efecto esperado |
|-------|-----------------|
| 1-5   | Cambio sutil, difícil de detectar |
| 10-20 | Efecto claro manteniendo coherencia — zona objetivo |
| > 30  | Output incoherente o degenerado |

In [7]:
ALPHA = 1

def steering_hook(value, hook):
    value[:, :, :] += ALPHA * steering_vector
    return value

def generate(prompt, max_new_tokens=30, use_steering=False):
    tokens = model.to_tokens(prompt)
    for _ in range(max_new_tokens):
        if use_steering:
            with model.hooks(fwd_hooks=[(f"blocks.{LAYER}.hook_resid_post", steering_hook)]):
                logits = model(tokens)
        else:
            logits = model(tokens)
        next_token = logits[0, -1, :].argmax(dim=-1, keepdim=True).unsqueeze(0)
        tokens = torch.cat([tokens, next_token], dim=1)
    return model.to_string(tokens[0])

## 6. Comparativa — baseline vs steered

**Qué buscar en los outputs:**

Un steering efectivo de calma/esperanza debería producir:
- Desaparecen palabras de miedo (*terrified, devastated, panic, overwhelmed*)
- Aparecen palabras de tranquilidad (*calm, hopeful, relieved, at peace, fine, okay*)
- El paciente reacciona con aceptación o serenidad ante la noticia

**Si el output steered es incoherente o repetitivo:** alpha demasiado alto. Reduce a 10 o prueba con 5.

**Si no ves ningún cambio:** alpha demasiado bajo o el vector no captura bien el concepto emocional en esa capa. Sube alpha o prueba la capa 8 o 12.

In [8]:
prompt = "The doctor told the patient that he had cancer. The patient felt"

baseline = generate(prompt, use_steering=False)
steered  = generate(prompt, use_steering=True)

print(f"PROMPT:   {prompt}")
print(f"BASELINE: {baseline}")
print(f"STEERED:  {steered}")

PROMPT:   The doctor told the patient that he had cancer. The patient felt
BASELINE: <|endoftext|>The doctor told the patient that he had cancer. The patient felt that he was being treated for cancer. The doctor told the patient that he had cancer.

The doctor told the patient that he had cancer.
STEERED:  <|endoftext|>The doctor told the patient that he had cancer. The patient felt better.

"I'm fine," he said. "I'm fine."

The doctor said he was fine.

"I


In [9]:
import torch
from transformer_lens import HookedTransformer

  # ── 1. Cargar modelo ──────────────────────────────────────────────
model = HookedTransformer.from_pretrained("gpt2")

  # ── 2. Construir el steering vector (prompting contrastivo) ───────
_, cache_pos = model.run_with_cache("The patient is happy and calm")
_, cache_neg = model.run_with_cache("The patient is scared and panicking")

  # Diferencia en el residual stream de la capa 8, último token
steering_vector = (
      cache_pos["resid_post", 8][0, -1, :] -
      cache_neg["resid_post", 8][0, -1, :]
  )

  # ── 3. Función de generación (con o sin hook) ─────────────────────
def generate(prompt, alpha=0, n_tokens=20):
      tokens = model.to_tokens(prompt)

      def hook_fn(activacion, hook):
          activacion[:, -1, :] += alpha * steering_vector
          return activacion

      for _ in range(n_tokens):
          if alpha == 0:
              logits = model(tokens)
          else:
              with model.hooks(fwd_hooks=[("blocks.8.hook_resid_post", hook_fn)]):
                  logits = model(tokens)
          next_token = logits[0, -1, :].argmax(dim=-1, keepdim=True).unsqueeze(0)
          tokens = torch.cat([tokens, next_token], dim=1)

      return model.to_string(tokens[0])

  # ── 4. Comparativa ────────────────────────────────────────────────
prompt = "The doctor told the patient that"

print("BASE:      ", generate(prompt, alpha=0))
print("ALPHA = 1: ", generate(prompt, alpha=1))
print("ALPHA = 3: ", generate(prompt, alpha=3))
print("ALPHA = 5: ", generate(prompt, alpha=5))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
BASE:       <|endoftext|>The doctor told the patient that he had been diagnosed with a rare form of cancer, but that he had not been diagnosed with the
ALPHA = 1:  <|endoftext|>The doctor told the patient that he was in a "good place" and that he was "going to be fine."


ALPHA = 3:  <|endoftext|>The doctor told the patient that the and-going-going-going-going-going-going-going-going-going
ALPHA = 5:  <|endoftext|>The doctor told the patient that enough-going-going-going-going-going-going-going-going-going-
